In [6]:
from gradio_client import Client   # for api use

import ast                         # for text to list conversion (make data useful)
import pandas as pd                # for easy data use; you can use polars or whatever you like

# start 8:14pm

# Accessing the API

Accessing the API is simple. You pass your key to the client, and then you will see a message the API has loaded.

In [7]:
token = '../api_creds/vi_key_1.csv'

# load my key
df = pd.read_csv(token) # easy and lazy lol; using pandas anyway ;p
api_key = df['key'][0]  # easy and lazy lol; using pandas anyway ;p

# connect to api
client = Client("verdantintel/gsv3", token=api_key)

Loaded as API: https://verdantintel-gsv3.hf.space


# Helper Function

In [10]:
def parse_events(text):

    events = []

    for block in text.strip().split("\n\n"):
        lines = block.split("\n")

        event = {"title": lines[0].strip()}

        for line in lines[1:]:
            if ":" in line:
                key, value = line.split(":", 1)
                key = key.strip().lower().replace(" ", "_")
                value = value.strip()
                event[key] = value

        events.append(event)

    return events

In [11]:
def ask_api(client, question):

    answer = client.predict(question=question, api_name="/predict")
    
    #answer = ast.literal_eval(answer)
    events = parse_events(answer)

    return events

    #return answer

# Searching for Data

In [12]:
question = 'what is happening in hillsboro, beaverton, or portland on March 18-22, 2026?'

answer = ask_api(client, question)
len(answer)

67

In [13]:
answer[0]

{'title': 'Johnny Franco and His Real Brother Dom - EP Release',
 'performers': 'Johnny Franco and His Real Brother Dom',
 'event_dates': 'March 20th 2026, 7:30 pm',
 'price': '$20.00',
 'venue': 'The Old Church Concert Hall',
 'location': '1422 SW 11th Ave, Portland, Oregon 97201',
 'contact_information': 'Not provided in source.',
 'description': "This event celebrates the EP release of 'At Bens Garage' by Brazilian duo Johnny Franco and his Real Brother Dom, marking their debut performance at The Old Church Concert Hall. The show will feature a selection of crowd-favorite covers and originals along with 30 minutes of new music.",
 'genres': 'Not provided in source.',
 'source_url': 'https://aftontickets.com/tocjohnnyfranco2026'}

# Methodology

- Fetch GrooveSeeker events (done)
- Read quickly with my eyes looking for interesting things
- Reduce to a few options that I am actually interest in
- Go have fun

The test of GrooveSeeker is whether or not it leads to things that I enjoy. The technical tests are already done.

In [52]:
answer[0:2]

[{'title': 'Johnny Franco and His Real Brother Dom - EP Release',
  'performers': 'Johnny Franco and His Real Brother Dom',
  'event_dates': 'March 20th 2026, 7:30 pm',
  'price': '$20.00',
  'venue': 'The Old Church Concert Hall',
  'location': '1422 SW 11th Ave, Portland, Oregon 97201',
  'contact_information': 'Not provided in source.',
  'description': "This event celebrates the EP release of 'At Bens Garage' by Brazilian duo Johnny Franco and his Real Brother Dom, marking their debut performance at The Old Church Concert Hall. The show will feature a selection of crowd-favorite covers and originals along with 30 minutes of new music.",
  'genres': 'Not provided in source.',
  'source_url': 'https://aftontickets.com/tocjohnnyfranco2026'},
 {'title': 'Magnolia Festival Garden Tea Tastings with Hannah Ma',
  'performers': 'Hannah Ma',
  'event_dates': '21st March 2026 11:00 AM – 12:00 PM',
  'price': '$36.95 – $55.95',
  'venue': 'Lan Su Chinese Garden',
  'location': '239 NW Everett

Let's loop through titles and see what looks interesting. I don't want to be distracted by the other context yet. Title and descriptiona re good enough.

In [56]:
df = pd.DataFrame(answer)

df = df[df['source_url'].str.startswith('https')] # simple URL filter; drop empties; the rest are https

df.drop_duplicates(inplace=True)
df.sort_values('title', inplace=True)

for row in df.iterrows():
    
    title = row[1]['title']
    description = row[1]['description']
    event_dates = row[1]['event_dates']
    
    url = row[1]['source_url']
    
    print(title)
    print()
    print(event_dates)
    print()
    print(description)
    print()
    print(url)
    print()
    print('-------------------------------------------------')
    print()

#PlantWednesday Spotlight

March 18, 2026, 11:00 AM – 11:30 AM

Join the horticulture team at Lan Su Chinese Garden for a 30-minute tour exploring a different plant each week, providing gardening tips and insights into the garden's operations.

https://lansugarden.org/event/plant-wednesday-spotlight/2026-03-18/

-------------------------------------------------

ANNA MOSS

March 20, 2026, Doors: 7:00 pm, Show: 8:00 pm PDT

Vocalist/multi-instrumentalist Anna Moss brings her sultry, stripped-down soul influenced by Southern R&B, Americana, and porch jazz to the Jack London Revue.

https://www.jacklondonrevue.com/tm-event/anna-moss/

-------------------------------------------------

Almost Famous Crafternoon w/ Ritual Dyes

March 22, 2026 4:00 pm - 6:00 pm

A crafting event hosted by the Portland Art Museum featuring Ritual Dyes.

https://portlandartmuseum.org/event/almost-famous-crafternoon-w-ritual-dyes/

-------------------------------------------------

BLACK PONTIAC (B.C.), SUNFISH

This is enough for my needs. I have event titles, descriptions, and links. With this, I'll copy/paste into a textfile and whittle this down to things that are actually interestingto me.

Produced 639 line long text file, a lot to look through. KISS and YAGNI. :)

Reduced to 260 lines, found a rave party and all kinds of cool stuff. Victory.

Now to shortlist what I want to go to and then go.